# Station Mapping Check

Checks how well station names referenced in `raw_data/raw_trips` (`start_station_name`, `end_station_name`) map onto the reference station table in `raw_data/stations.parquet` (`name`).

Goals:
- Quantify exact-match coverage between trip station names and reference station names.
- Surface names that appear in trips but are missing from the station reference table (and vice versa).
- Distinguish trivial formatting mismatches (whitespace/case) from true naming mismatches.
- Report the row-level match rate (what share of trip rows have both endpoints resolvable to a known station).

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Notebook lives in notebooks/, so the project root is one level up.
ROOT = Path.cwd().parent
RAW_DATA = ROOT / "raw_data"
STATIONS_PATH = RAW_DATA / "stations.parquet"
RAW_TRIPS_PATH = RAW_DATA / "raw_trips"

print(f"Project root: {ROOT}")
print(f"Stations path: {STATIONS_PATH} (exists={STATIONS_PATH.exists()})")
print(f"Raw trips path: {RAW_TRIPS_PATH} (exists={RAW_TRIPS_PATH.exists()})")

Project root: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor
Stations path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\stations.parquet (exists=True)
Raw trips path: c:\Users\atlas\OneDrive\Documents\Portfolio Projects\Capital Bike Share Projects\02 Capital BikeShare Data Preprocessor\raw_data\raw_trips (exists=True)


## Load data

- `stations`: reference station table (`raw_data/stations.parquet`), columns include `station_id`, `short_name`, `name`, `lat`, `lon`, `capacity`.
- `trips`: trip data (`raw_data/raw_trips/`), read as a full partitioned parquet dataset so any additional `year=/month=` partitions synced later are picked up automatically. Columns include `start_station_name`, `end_station_name`.

In [ ]:
stations = pd.read_parquet(STATIONS_PATH)
trips = pd.read_parquet(
    RAW_TRIPS_PATH,
    columns=["start_station_name", "end_station_name", "period"],
)

print(f"stations: {stations.shape[0]:,} rows, columns: {list(stations.columns)}")
print(f"trips:    {trips.shape[0]:,} rows, columns: {list(trips.columns)}")
print(f"Periods covered in trips: {sorted(trips['period'].dropna().unique())}")

stations.head()

stations: 866 rows, columns: ['station_id', 'short_name', 'name', 'lat', 'lon', 'capacity', 'updated_at']
trips:    22,047,677 rows, columns: ['start_station_name', 'end_station_name', 'period']
Periods covered in trips: ['2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018-01', '2018-02', '2018-03', '2018-04', '2018-05', '2018-06', '2018-07', '2018-10', '2026-01', '2026-02']


,station_id,short_name,name,lat,lon,capacity,updated_at
0,2070121870129475816,32324,Scotts Crossing Rd & Spring Gate Dr,38.927018,-77.211094,11,2026-09-26T00:50:25
1,0825be94-1f3f-11e7-bf6b-3863bb334450,31072,31st St & S Woodrow St,38.837846,-77.093478,8,2026-09-26T00:50:25
2,2254169244471500614,32367,Banneker Pool,38.922301,-77.023405,14,2026-09-26T00:50:25
3,2113890246552748136,32304,Westwood Center Dr & Sheraton Tysons Dr,38.929129,-77.246523,12,2026-09-26T00:50:25
4,082569e6-1f3f-11e7-bf6b-3863bb334450,31066,Columbia Pike & S Orme St,38.867217,-77.072296,8,2026-09-26T00:50:25


## Build the name universes

Note the naming mismatch: the station reference table uses `name`, while trips use `start_station_name` / `end_station_name`. We treat all three as the same logical "station name" concept for comparison purposes.

In [ ]:
station_names = set(stations["name"].dropna().unique())
start_names = set(trips["start_station_name"].dropna().unique())
end_names = set(trips["end_station_name"].dropna().unique())
trip_names = start_names | end_names

print(f"Unique station names (stations.name):        {len(station_names):,}")
print(f"Unique start_station_name values:             {len(start_names):,}")
print(f"Unique end_station_name values:               {len(end_names):,}")
print(f"Unique trip station names (start | end):      {len(trip_names):,}")

Unique station names (stations.name):        866
Unique start_station_name values:             970
Unique end_station_name values:               967
Unique trip station names (start | end):      971


## Exact-match coverage

How many trip station names have an exact match in the station reference table, and how many don't?

In [ ]:
matched_names = trip_names & station_names
trip_only_names = trip_names - station_names  # in trips, missing from stations
station_only_names = station_names - trip_names  # in stations, never used in trips

coverage_rate = len(matched_names) / len(trip_names) if trip_names else float("nan")

print(f"Trip names matched exactly to a station:      {len(matched_names):,}")
print(f"Trip names with NO match in stations:         {len(trip_only_names):,}")
print(f"Station names never referenced in trips:      {len(station_only_names):,}")
print(f"Exact-match coverage rate (trip names):       {coverage_rate:.2%}")

Trip names matched exactly to a station:      812
Trip names with NO match in stations:         159
Station names never referenced in trips:      54
Exact-match coverage rate (trip names):       83.63%


### Trip station names missing from the station reference table

These are `start_station_name` / `end_station_name` values that don't exactly match any `stations.name` value — potential data quality issues, renamed/retired stations, or naming inconsistencies.

In [ ]:
trip_only_df = pd.DataFrame(sorted(trip_only_names), columns=["trip_station_name"])
print(f"{len(trip_only_df):,} trip station names have no exact match in stations.parquet")
trip_only_df.head(50)

159 trip station names have no exact match in stations.parquet


,trip_station_name
0,10th & Florida Ave NW
1,11th & H St NE
2,12th & Army Navy Dr
3,14th & D St NW / Ronald Reagan Building
4,14th & Luzon St NW
5,14th & Rhode Island Ave NW
6,14th St & Colorado Ave NW
7,14th St & Spring Rd NW
8,15th & Crystal Dr
9,15th & Euclid St NW


### Reference stations never referenced in trips

These may be legitimate (e.g. stations added after the trip periods currently loaded, or stations with zero activity in the loaded periods) rather than data quality issues.

In [ ]:
station_only_df = pd.DataFrame(sorted(station_only_names), columns=["station_name"])
print(f"{len(station_only_df):,} reference stations are never referenced in the loaded trip data")
station_only_df.head(50)

54 reference stations are never referenced in the loaded trip data


,station_name
0,12th St N & Fort Myer Dr.
1,14th & T St NW
2,18th & Swann St NW
3,18th & Willard St NW
4,31st & K St NW
5,31st & M St NW
6,4th & Q St NW
7,4th St & Jefferson Dr SW
8,Adams Mill Rd & Ontario Pl NW Walter Pierce Park
9,Addison Road Metro Capital Bikeshare


## Normalized-name re-check

Some "mismatches" above may just be formatting differences (leading/trailing whitespace, casing). Re-run the comparison after normalizing (`strip` + `casefold`) to separate trivial formatting issues from true naming mismatches.

In [ ]:
def normalize(name: str) -> str:
    return name.strip().casefold()


norm_station_names = {normalize(n) for n in station_names}
norm_trip_only_names = {normalize(n) for n in trip_only_names}

# Of the names that failed exact match, how many are recoverable via normalization?
recovered_by_normalization = norm_trip_only_names & norm_station_names
still_unmatched_normalized = norm_trip_only_names - norm_station_names

print(f"Previously unmatched trip names:                              {len(trip_only_names):,}")
print(f"...recoverable after strip+casefold normalization:            {len(recovered_by_normalization):,}")
print(f"...still unmatched after normalization (true mismatches):     {len(still_unmatched_normalized):,}")

true_mismatch_df = pd.DataFrame(sorted(still_unmatched_normalized), columns=["normalized_trip_station_name"])
true_mismatch_df.head(50)

Previously unmatched trip names:                              159
...recoverable after strip+casefold normalization:            0
...still unmatched after normalization (true mismatches):     159


,normalized_trip_station_name
0,10th & florida ave nw
1,11th & h st ne
2,12th & army navy dr
3,14th & d st nw / ronald reagan building
4,14th & luzon st nw
5,14th & rhode island ave nw
6,14th st & colorado ave nw
7,14th st & spring rd nw
8,15th & crystal dr
9,15th & euclid st nw


## Closest-match suggestions for true mismatches

For trip station names that still don't match after normalization, use string similarity (`difflib.get_close_matches`, stdlib — no extra dependency) against the reference station names to suggest likely intended matches (e.g. typos, abbreviations, minor renames). This doesn't auto-fix anything — it's a shortlist to review manually.

In [ ]:
from difflib import SequenceMatcher, get_close_matches

# Map normalized station name -> a representative original station name (for display).
norm_to_station_display = {}
for original in station_names:
    norm_to_station_display.setdefault(normalize(original), original)

SIMILARITY_CUTOFF = 0.6  # difflib ratio threshold; lower = more (looser) suggestions
MAX_SUGGESTIONS = 3

suggestions = []
for trip_name in sorted(trip_only_names):
    norm_trip_name = normalize(trip_name)
    if norm_trip_name in norm_station_names:
        continue  # recovered by normalization already; not a "true" mismatch

    close = get_close_matches(
        norm_trip_name,
        norm_station_names,
        n=MAX_SUGGESTIONS,
        cutoff=SIMILARITY_CUTOFF,
    )
    if not close:
        suggestions.append(
            {
                "trip_station_name": trip_name,
                "suggested_station_name": None,
                "similarity": None,
            }
        )
        continue

    for candidate in close:
        similarity = SequenceMatcher(None, norm_trip_name, candidate).ratio()
        suggestions.append(
            {
                "trip_station_name": trip_name,
                "suggested_station_name": norm_to_station_display[candidate],
                "similarity": round(similarity, 3),
            }
        )

suggestions_df = pd.DataFrame(suggestions)
suggestions_df = suggestions_df.sort_values(
    ["similarity", "trip_station_name"], ascending=[False, True], na_position="last"
)

n_with_suggestion = suggestions_df.loc[suggestions_df["suggested_station_name"].notna(), "trip_station_name"].nunique()
n_no_suggestion = suggestions_df.loc[suggestions_df["suggested_station_name"].isna(), "trip_station_name"].nunique()

print(f"True mismatches with at least one close-match suggestion: {n_with_suggestion:,}")
print(f"True mismatches with NO close match found (cutoff={SIMILARITY_CUTOFF}): {n_no_suggestion:,}")
suggestions_df.head(50)

True mismatches with at least one close-match suggestion: 134
True mismatches with NO close match found (cutoff=0.6): 25


,trip_station_name,suggested_station_name,similarity
115,Blueridge Ave & Elkins St,Blueridge Ave & Elkin St,0.980
174,Friendship Hts Metro/Wisconsin Ave & Wisconsin Cir,Friendship Hts Metro / Wisconsin Ave & Wisconsin Cir,0.980
162,East Falls Church Metro/Sycamore St & 19th St N,East Falls Church Metro / Sycamore St & 19th St N,0.979
235,Montgomery College/W Campus Dr & Mannakee St,Montgomery College / W Campus Dr & Mannakee St,0.978
319,Silver Spring Metro/Colesville Rd & Wayne Ave,Silver Spring Metro / Colesville Rd & Wayne Ave,0.978
348,Virginia Square Metro / N Monroe St & 9th St N,Virginia Square Metro / Monroe St & 9th St N,0.978
213,Long Bridge Park/Long Bridge Dr & 6th St S,Long Bridge Park / Long Bridge Dr & 6th St S,0.977
126,Central Library / N Quincy St & 10th St N,Central Library / Quincy St & 10th St N,0.975
25,15th & Euclid St NW,15th & Euclid St NW,0.974
353,Wheaton Metro/Georgia Ave & Reedie Dr,Wheaton Metro / Georgia Ave & Reedie Dr,0.974


### Best single suggestion per unmatched name

Collapse to the top candidate per trip station name, ranked by similarity, for a quick reviewable shortlist.

In [ ]:
best_suggestion_df = (
    suggestions_df.dropna(subset=["suggested_station_name"])
    .sort_values("similarity", ascending=False)
    .drop_duplicates(subset=["trip_station_name"], keep="first")
    .sort_values(["similarity", "trip_station_name"], ascending=[False, True])
    .reset_index(drop=True)
)

no_suggestion_names = sorted(
    set(suggestions_df.loc[suggestions_df["suggested_station_name"].isna(), "trip_station_name"])
)

print(f"Best-guess suggestions available for {len(best_suggestion_df):,} unmatched trip station names")
print(f"No candidate found above cutoff={SIMILARITY_CUTOFF} for {len(no_suggestion_names):,} names")
best_suggestion_df

Best-guess suggestions available for 134 unmatched trip station names
No candidate found above cutoff=0.6 for 25 names


,trip_station_name,suggested_station_name,similarity
0,Blueridge Ave & Elkins St,Blueridge Ave & Elkin St,0.980
1,Friendship Hts Metro/Wisconsin Ave & Wisconsin Cir,Friendship Hts Metro / Wisconsin Ave & Wisconsin Cir,0.980
2,East Falls Church Metro/Sycamore St & 19th St N,East Falls Church Metro / Sycamore St & 19th St N,0.979
3,Montgomery College/W Campus Dr & Mannakee St,Montgomery College / W Campus Dr & Mannakee St,0.978
4,Silver Spring Metro/Colesville Rd & Wayne Ave,Silver Spring Metro / Colesville Rd & Wayne Ave,0.978
...,...,...,...
129,27th & Crystal Dr,7th & S St NW,0.600
130,Army Navy Dr & S Eads St,Fairfax Dr & N Randolph St,0.600
131,Army Navy Dr & S Nash St,Fairfax Dr & N Randolph St,0.600
132,Eastern Market Metro / Pennsylvania Ave & 7th St SE,Potomac & Pennsylvania Ave SE,0.600


## Start vs. end name consistency

Sanity check: does every station name that appears as a `start_station_name` also appear somewhere as an `end_station_name`, and vice versa? Large asymmetries could indicate data issues rather than genuine one-directional station usage.

In [ ]:
start_only = start_names - end_names
end_only = end_names - start_names

print(f"Names used as start_station_name only (never as end): {len(start_only):,}")
print(f"Names used as end_station_name only (never as start): {len(end_only):,}")

if start_only:
    print("\nSample start-only names:")
    for n in sorted(start_only)[:10]:
        print(f"  - {n}")

if end_only:
    print("\nSample end-only names:")
    for n in sorted(end_only)[:10]:
        print(f"  - {n}")

Names used as start_station_name only (never as end): 4
Names used as end_station_name only (never as start): 1

Sample start-only names:
  - 6035 Warehouse
  - 9th & F St NW
  - Stewart & April
  - Stewart Ln & Old Columbia Pike

Sample end-only names:
  - White Oak Rec Center


## Row-level match rate

Unique-name coverage doesn't tell us how much of the actual trip *volume* is affected. Compute, per row, whether `start_station_name` and `end_station_name` each resolve to a known station, and report the share of rows fully matched on both ends.

In [ ]:
total_rows = len(trips)

start_matched = trips["start_station_name"].isin(station_names)
end_matched = trips["end_station_name"].isin(station_names)
both_matched = start_matched & end_matched

start_match_rate = start_matched.mean()
end_match_rate = end_matched.mean()
both_match_rate = both_matched.mean()

print(f"Total trip rows:                                  {total_rows:,}")
print(f"Rows with start_station_name matched to a station: {start_matched.sum():,} ({start_match_rate:.2%})")
print(f"Rows with end_station_name matched to a station:   {end_matched.sum():,} ({end_match_rate:.2%})")
print(f"Rows with BOTH endpoints matched:                  {both_matched.sum():,} ({both_match_rate:.2%})")
print(f"Rows with at least one endpoint unmatched:         {(~both_matched).sum():,} ({(~both_matched).mean():.2%})")

Total trip rows:                                  22,047,677
Rows with start_station_name matched to a station: 17,438,186 (79.09%)
Rows with end_station_name matched to a station:   17,297,037 (78.45%)
Rows with BOTH endpoints matched:                  14,102,665 (63.96%)
Rows with at least one endpoint unmatched:         7,945,012 (36.04%)


### Which unmatched names account for the most rows?

Ranks unmatched station names (start or end) by how many trip rows they appear on, so we can prioritize investigating the ones with the biggest impact.

In [ ]:
start_counts = (
    trips.loc[~start_matched, "start_station_name"]
    .value_counts()
    .rename("as_start")
)
end_counts = (
    trips.loc[~end_matched, "end_station_name"]
    .value_counts()
    .rename("as_end")
)

unmatched_impact = (
    pd.concat([start_counts, end_counts], axis=1)
    .fillna(0)
    .astype(int)
)
unmatched_impact["total_rows"] = unmatched_impact["as_start"] + unmatched_impact["as_end"]
unmatched_impact = unmatched_impact.sort_values("total_rows", ascending=False)

print(f"{len(unmatched_impact):,} distinct unmatched names appear across trip rows")
print(f"They account for {unmatched_impact['total_rows'].sum():,} row-endpoint occurrences")
unmatched_impact.head(20)

159 distinct unmatched names appear across trip rows
They account for 9,196,018 row-endpoint occurrences


,as_start,as_end,total_rows
Massachusetts Ave & Dupont Circle NW,419819,469745,889564
Eastern Market Metro / Pennsylvania Ave & 7th St SE,243598,238346,481944
Smithsonian-National Mall / Jefferson Dr & 12th St SW,208666,216666,425332
14th & Rhode Island Ave NW,207577,217375,424952
New York Ave & 15th St NW,183942,192019,375961
21st & M St NW,177542,190180,367722
M St & Pennsylvania Ave NW,143449,175641,319090
14th & D St NW / Ronald Reagan Building,138643,159223,297866
7th & T St NW,141581,148296,289877
C & O Canal & Wisconsin Ave NW,124989,164670,289659


## Summary

In [ ]:
from IPython.display import Markdown, display

summary = f"""
| Metric | Value |
|---|---|
| Reference stations (`stations.name`) | {len(station_names):,} |
| Unique trip station names (start ∪ end) | {len(trip_names):,} |
| Exact-match coverage (unique names) | {coverage_rate:.2%} |
| Trip names unmatched (exact) | {len(trip_only_names):,} |
| ...still unmatched after normalization | {len(still_unmatched_normalized):,} |
| ...with a close-match suggestion (similarity ≥ {SIMILARITY_CUTOFF}) | {len(best_suggestion_df):,} |
| ...with no close match found | {len(no_suggestion_names):,} |
| Reference stations never used in trips | {len(station_only_names):,} |
| Trip rows | {total_rows:,} |
| Row-level match rate (both endpoints) | {both_match_rate:.2%} |
"""

display(Markdown(summary))


| Metric | Value |
|---|---|
| Reference stations (`stations.name`) | 866 |
| Unique trip station names (start ∪ end) | 971 |
| Exact-match coverage (unique names) | 83.63% |
| Trip names unmatched (exact) | 159 |
| ...still unmatched after normalization | 159 |
| ...with a close-match suggestion (similarity ≥ 0.6) | 134 |
| ...with no close match found | 25 |
| Reference stations never used in trips | 54 |
| Trip rows | 22,047,677 |
| Row-level match rate (both endpoints) | 63.96% |
